# Bab 19: Jaringan Bayes

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab19/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab19")
print("Folder kerja:", os.getcwd())

## Listing 19.1: Memeriksa tiga pola independensi pada data biji

`kode/bab19/independensi.py`

In [ ]:
import sys; sys.path.insert(0, "kode/bab19")
import pandas as pd
from jb import muat_biji

d = muat_biji()
def porsi(df, kolom, target, nilai="ya"):
    kena = df[target] == nilai
    return kena.groupby([df[c] for c in kolom]).mean().round(3)

print("1) P(cacat=ya | musim): musim dan cacat bergantung")
print(porsi(d, ["musim"], "cacat").to_string())
print("\n2) P(cacat=ya | jamur, jemur, musim): bila jamur dan")
print("   jemur diketahui, musim hampir tak menambah informasi")
t = porsi(d, ["jamur", "jemur", "musim"], "cacat").unstack()
print(t.to_string())
print("\n3) kolider: lama simpan -> jamur <- kadar air")
m = pd.read_csv("kode/data/mutu_biji.csv")
def r(df): return df.kadar_air.corr(df.lama_simpan_hari)
jm = m[m.jamur == 1]
print("   korelasi kadar air dan lama simpan")
print(f"   semua kiriman    (n={len(m)}): {r(m):+.3f}")
print(f"   kiriman berjamur (n={len(jm)}) : {r(jm):+.3f}")

## Listing 19.2: Modul jaringan Bayes diskret

`kode/bab19/jb.py`

In [ ]:
"""Jaringan Bayes diskret kecil: CPT, inferensi enumerasi, intervensi.

Cukup untuk belasan variabel diskret; untuk jaringan besar gunakan
pustaka seperti pgmpy.
"""
import itertools

import numpy as np
import pandas as pd


def muat_biji(path="kode/data/mutu_biji.csv"):
    """Diskretkan data mutu biji menjadi tujuh variabel kategorik."""
    d = pd.read_csv(path, parse_dates=["tanggal"])
    ya = lambda b, s1, s0: np.where(b, s1, s0)
    x = pd.DataFrame({
        "musim": d.musim_panen,
        "jemur": d.cara_jemur,
        "basah": ya(d.proses == "giling basah", "ya", "tidak"),
        "simpan": ya(d.lama_simpan_hari > 60, "lama", "singkat"),
        "air": ya(d.kadar_air > 12.5, "tinggi", "normal"),
        "jamur": ya(d.jamur == 1, "ya", "tidak"),
        "cacat": ya(d.cacat_primer > 5, "ya", "tidak"),
    })
    x["tanggal"] = d.tanggal
    return x


class JaringanBayes:
    def __init__(self, induk, nilai):
        self.induk = induk            # {simpul: [induk, ...]}
        self.nilai = nilai            # {simpul: [nilai yang mungkin]}
        self.cpt = {}

    def latih(self, data, alfa=1.0):
        """Taksir CPT: hitungan frekuensi + pemulusan Laplace."""
        for v, pa in self.induk.items():
            nilai_pa = [self.nilai[p] for p in pa]
            tabel = {}
            for k in itertools.product(*nilai_pa):
                bagian = data
                for p, nilai_p in zip(pa, k):
                    bagian = bagian[bagian[p] == nilai_p]
                hit = bagian[v].value_counts()
                n = np.array([hit.get(s, 0) for s in self.nilai[v]])
                n = n + alfa
                tabel[k] = dict(zip(self.nilai[v], n / n.sum()))
            self.cpt[v] = tabel
        return self

    def gabungan(self, intervensi=None):
        """Distribusi gabungan semua variabel sebagai DataFrame.

        intervensi={v: nilai} memotong panah yang masuk ke v dan
        menetapkan nilainya (operator do).
        """
        intervensi = intervensi or {}
        simpul = list(self.induk)
        baris = []
        semua = [self.nilai[v] for v in simpul]
        for konf in itertools.product(*semua):
            a = dict(zip(simpul, konf))
            p = 1.0
            for v in simpul:
                if v in intervensi:
                    p *= 1.0 if a[v] == intervensi[v] else 0.0
                else:
                    k = tuple(a[q] for q in self.induk[v])
                    p *= self.cpt[v][k][a[v]]
                if p == 0:
                    break
            a["p"] = p
            baris.append(a)
        return pd.DataFrame(baris)

    def tanya(self, target, bukti=None, intervensi=None):
        """P(target | bukti) atau P(target | do(iv), bukti)."""
        g = self.gabungan(intervensi)
        for v, s in (bukti or {}).items():
            g = g[g[v] == s]
        return g.groupby(target).p.sum() / g.p.sum()

    def log_lik(self, data):
        ll = 0.0
        for v, pa in self.induk.items():
            for _, r in data.iterrows():
                k = tuple(r[q] for q in pa)
                ll += np.log(self.cpt[v][k][r[v]])
        return ll

## Listing 19.3: Inferensi pada jaringan mini

`kode/bab19/hitung_mini.py`

In [ ]:
import sys; sys.path.insert(0, "kode/bab19")
from jb import JaringanBayes

Y, T = "ya", "tidak"
jb = JaringanBayes(
    induk={"H": [], "P": [], "K": ["H", "P"], "J": ["K"], "C": ["J"]},
    nilai={v: [Y, T] for v in "HPKJC"})
def biner(p): return {Y: p, T: 1 - p}
jb.cpt = {
    "H": {(): biner(.4)},
    "P": {(): biner(.4)},
    "K": {(Y, Y): biner(.7), (Y, T): biner(.4),
          (T, Y): biner(.3), (T, T): biner(.1)},
    "J": {(Y,): biner(.5), (T,): biner(.05)},
    "C": {(Y,): biner(.8), (T,): biner(.1)},
}
def p(target, **bukti): return jb.tanya(target, bukti)[Y]

print(f"P(C)               = {p('C'):.4f}")
print(f"P(C | H)           = {p('C', H=Y):.4f}   prediksi")
print(f"P(H | C)           = {p('H', C=Y):.4f}   diagnosis")
print("bukti berurutan:")
print(f"  P(H)             = {p('H'):.4f}")
print(f"  P(H | C)         = {p('H', C=Y):.4f}")
print(f"  P(H | C,K)       = {p('H', C=Y, K=Y):.4f}")
print(f"  P(H | C,K,P)     = {p('H', C=Y, K=Y, P=Y):.4f}"
      "   explaining away")
print(f"  P(H | C,K,P,J)   = {p('H', C=Y, K=Y, P=Y, J=Y):.4f}"
      "   J tak menambah")
print(f"P(H | K)           = {p('H', K=Y):.4f}   = P(H | C,K)")
print(f"P(J | C)           = {p('J', C=Y):.4f}")

## Listing 19.4: Menaksir CPT dari data

`kode/bab19/belajar_parameter.py`

In [ ]:
import sys; sys.path.insert(0, "kode/bab19")
import pandas as pd
from jb import JaringanBayes, muat_biji

d = muat_biji()
hit = pd.crosstab([d.air, d.simpan], d.jamur)
print("hitungan kiriman menurut induk jamur:")
print(hit.to_string())
mle = hit["ya"] / hit.sum(1)
lap = (hit["ya"] + 1) / (hit.sum(1) + 2)
print("\nP(jamur=ya | air, simpan):")
tabel = pd.DataFrame({"MLE": mle, "Laplace": lap}).round(3)
print(tabel.to_string())

print("\nCPT cacat dengan dua induk (jamur, jemur):")
V = ["jamur", "jemur", "cacat"]
jb = JaringanBayes({"jamur": [], "jemur": [],
                    "cacat": ["jamur", "jemur"]},
                   {v: sorted(d[v].unique()) for v in V}).latih(d)
for (jm, je), t in jb.cpt["cacat"].items():
    n = ((d.jamur == jm) & (d.jemur == je)).sum()
    print(f"  jamur={jm:5s} jemur={je:9s} n={n:4d}"
          f"  P(cacat=ya)={t['ya']:.3f}")

## Listing 19.5: Pendakian bukit dengan skor BIC

`kode/bab19/belajar_struktur.py`

In [ ]:
import sys; sys.path.insert(0, "kode/bab19")
import numpy as np
from jb import muat_biji

d = muat_biji().drop(columns="tanggal")
V = list(d.columns)
N = len(d)

def skor_lokal(v, pa):
    """BIC satu simpul: log-likelihood dikurangi penalti."""
    r = d[v].nunique()
    if pa:
        hit = d.groupby(list(pa) + [v]).size().unstack(fill_value=0)
        q = int(np.prod([d[p].nunique() for p in pa]))
    else:
        hit = d[v].value_counts().to_frame().T
        q = 1
    n = hit.to_numpy(float)
    nj = n.sum(1, keepdims=True)
    ll = (n * np.log(np.where(n > 0, n / nj, 1))).sum()
    return ll - 0.5 * np.log(N) * (r - 1) * q

def ada_siklus(induk):
    kunjung, tumpuk = set(), set()
    def dfs(v):
        if v in tumpuk: return True
        if v in kunjung: return False
        kunjung.add(v); tumpuk.add(v)
        hasil = any(dfs(p) for p in induk[v])
        tumpuk.discard(v)
        return hasil
    return any(dfs(v) for v in induk)

def calon_langkah(induk, a, b, boleh, maks):
    """Tambah a->b, atau hapus/balik a->b bila sudah ada."""
    if a in induk[b]:
        tanpa = tuple(p for p in induk[b] if p != a)
        yield {b: tanpa}
        if boleh(b, a) and len(induk[a]) < maks:
            yield {b: tanpa, a: induk[a] + (b,)}
    elif boleh(a, b) and len(induk[b]) < maks:
        yield {b: induk[b] + (a,)}

def mendaki_bukit(boleh=lambda a, b: True, maks=3):
    induk = {v: () for v in V}
    skor = {v: skor_lokal(v, ()) for v in V}
    while True:
        terbaik, langkah = 1e-9, None
        for a in V:
            for b in V:
                if a == b: continue
                for ubah in calon_langkah(induk, a, b, boleh, maks):
                    if ada_siklus({**induk, **ubah}): continue
                    beda = sum(skor_lokal(v, pa) - skor[v]
                               for v, pa in ubah.items())
                    if beda > terbaik:
                        terbaik, langkah = beda, ubah
        if langkah is None:
            return induk, sum(skor.values())
        for v, pa in langkah.items():
            induk[v] = tuple(sorted(pa))
            skor[v] = skor_lokal(v, induk[v])

def cetak(induk, skor):
    print(f"  BIC = {skor:.1f}")
    for v in V:
        if induk[v]:
            print(f"  {' , '.join(induk[v]):>20s} -> {v}")

print("tanpa pengetahuan domain:")
cetak(*mendaki_bukit())
lapis = {"musim": 0, "jemur": 0, "basah": 0, "simpan": 0,
         "air": 1, "jamur": 2, "cacat": 3}
print("dengan urutan waktu (lapis):")
cetak(*mendaki_bukit(boleh=lambda a, b: lapis[a] < lapis[b]))

## Listing 19.6: Studi kasus sebab-akibat cacat biji

`kode/bab19/studi_kasus.py`

In [ ]:
import sys; sys.path.insert(0, "kode/bab19")
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from jb import JaringanBayes, muat_biji

d = muat_biji()
latih = d[d.tanggal < "2025-07-01"]
uji = d[d.tanggal >= "2025-07-01"]
print(f"latih {len(latih)} kiriman, uji {len(uji)} kiriman; "
      f"porsi cacat latih {np.mean(latih.cacat == 'ya'):.3f}")
NILAI = {v: sorted(d[v].unique()) for v in d.columns
         if v != "tanggal"}
AKAR = {"musim": [], "jemur": [], "basah": [], "simpan": []}
STRUKTUR = {
    "kosong": {v: [] for v in NILAI},
    "dipelajari": {**AKAR, "air": ["musim", "jemur", "basah"],
                   "jamur": ["air", "simpan"], "cacat": ["jamur"]},
    "pakar": {**AKAR, "air": ["musim", "jemur", "basah"],
              "jamur": ["air", "simpan"],
              "cacat": ["jamur", "jemur"]},
}
print("\nlog-likelihood rata-rata per kiriman uji:")
model = {}
for nama, induk in STRUKTUR.items():
    model[nama] = JaringanBayes(induk, NILAI).latih(latih)
    print(f"  {nama:10s} {model[nama].log_lik(uji) / len(uji):.3f}")

jb = model["pakar"]
gab = jb.gabungan()
TERIMA = ["musim", "jemur", "basah", "simpan", "air"]
def p_cacat(baris):
    g = gab
    for v in TERIMA:
        g = g[g[v] == baris[v]]
    return g.p[g.cacat == "ya"].sum() / g.p.sum()
p_bn = uji.apply(p_cacat, axis=1)
X = pd.get_dummies(d[TERIMA], drop_first=True).astype(float)
lr = LogisticRegression(max_iter=1000)
lr.fit(X.loc[latih.index], latih.cacat == "ya")
p_lr = lr.predict_proba(X.loc[uji.index])[:, 1]
y = (uji.cacat == "ya").to_numpy()
print(f"\nramalan cacat saat penerimaan (uji, {y.sum()} cacat):")
for nama, p in [("jaringan Bayes", p_bn), ("regresi logistik", p_lr)]:
    print(f"  {nama:17s} AUC {roc_auc_score(y, p):.3f}"
          f"  Brier {brier_score_loss(y, p):.4f}")

print("\nP(cacat) di bawah intervensi:")
dasar = jb.tanya("cacat")["ya"]
SKENARIO = {"tanpa intervensi": {},
            "do(jemur=mesin)": {"jemur": "mesin"},
            "do(simpan=singkat)": {"simpan": "singkat"},
            "do(air=normal)": {"air": "normal"},
            "do(air=normal, simpan=singkat)":
                {"air": "normal", "simpan": "singkat"}}
hasil = {}
for nama, iv in SKENARIO.items():
    hasil[nama] = h = jb.tanya("cacat", intervensi=iv)["ya"]
    print(f"  {nama:31s} {h:.3f}  ({h - dasar:+.3f})")

print("\nmengamati versus mengintervensi kadar air:")
for s in ["tinggi", "normal"]:
    amati = jb.tanya("cacat", bukti={"air": s})["ya"]
    ubah = jb.tanya("cacat", intervensi={"air": s})["ya"]
    print(f"  air={s:6s}  P(cacat | air) {amati:.3f}"
          f"   P(cacat | do(air)) {ubah:.3f}")

print("\ndiagnosis kiriman cacat: P(penyebab | cacat=ya)")
for v, s in [("air", "tinggi"), ("jamur", "ya"), ("jemur", "terpal"),
             ("simpan", "lama"), ("musim", "hujan")]:
    pri = jb.tanya(v)[s]
    pos = jb.tanya(v, bukti={"cacat": "ya"})[s]
    print(f"  {v + '=' + s:13s} prior {pri:.3f}"
          f" -> posterior {pos:.3f}")

KIRIMAN, RUGI = 500, 6_000_000
print(f"\nperkiraan kiriman cacat per tahun (dari {KIRIMAN}):")
for nama in ["tanpa intervensi", "do(air=normal)",
             "do(air=normal, simpan=singkat)"]:
    k = KIRIMAN * hasil[nama]
    print(f"  {nama:31s} {k:5.1f} kiriman,"
          f" rugi Rp{k * RUGI / 1e6:4.0f} jt")